# Placement Prediction Using Machine Learning

This notebook contains the complete pipeline for the Placement Prediction project, including Data Exploration, Preprocessing, Model Training, and Evaluation.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Set seaborn style for better visualization
sns.set_theme(style="whitegrid")

## 1. Load the Dataset

In [ ]:
df = pd.read_csv('placement_data.csv')
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

## 2. Exploratory Data Analysis (EDA)
Let's visualize the relationship between different features and the placement status.

In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(x='Placement_Status', data=df, palette='viridis')
plt.title('Distribution of Placement Status')
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
sns.boxplot(x='Placement_Status', y='CGPA', data=df, palette='Set2')
plt.title('CGPA vs Placement Status')
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
sns.boxplot(x='Placement_Status', y='Aptitude_Score', data=df, palette='Set3')
plt.title('Aptitude Score vs Placement Status')
plt.show()

## 3. Data Preprocessing

In [ ]:
# Encode categorical variables
le = LabelEncoder()
df['Extracurricular_Activities'] = le.fit_transform(df['Extracurricular_Activities'])
df['Placement_Training'] = le.fit_transform(df['Placement_Training'])

# Encode target variable
df['Placement_Status'] = df['Placement_Status'].map({'Placed': 1, 'Not Placed': 0})

df.head()

In [ ]:
# Split the data into features (X) and target (y)
X = df.drop('Placement_Status', axis=1)
y = df['Placement_Status']

# Train-test split (80% training, 20% testing)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scale the numerical features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## 4. Model Training and Evaluation

In [ ]:
def evaluate_model(model, X_test, y_test, model_name):
    y_pred = model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    print(f"--- {model_name} ---")
    print(f"Accuracy: {accuracy:.4f}\n")
    print("Classification Report:")
    print(classification_report(y_test, y_pred))
    
    # Plot Confusion Matrix
    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(4, 3))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
    plt.title(f'Confusion Matrix - {model_name}')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.show()

### Logistic Regression

In [ ]:
lr_model = LogisticRegression(random_state=42)
lr_model.fit(X_train_scaled, y_train)
evaluate_model(lr_model, X_test_scaled, y_test, "Logistic Regression")

### Random Forest Classifier

In [ ]:
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train_scaled, y_train)
evaluate_model(rf_model, X_test_scaled, y_test, "Random Forest Classifier")

## 5. Feature Importance

In [ ]:
feature_importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': rf_model.feature_importances_
}).sort_values(by='Importance', ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(x='Importance', y='Feature', data=feature_importance, palette='mako')
plt.title('Feature Importance (Random Forest)')
plt.show()